In [1]:
from model.LightGCN import *
from preprocess.AmazonBook import *
from evaluation.LightGCN_SISA import *
pd.options.display.max_rows = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
path = './dataset/amazon-book'
dataset = AmazonBook(path)

data = dataset.get()
num_users, num_books = dataset.getNumber()
config = {
    'k': 20,
    'lr': 0.001,
    'epochs': 1000,
    'num_layers': 2,
    'batch_size': 8192,
    'embedding_dim': 64,
    'num_users': num_users,
    'num_books': num_books,
    'tuning_type': None,
}
model = LightGCN(
    num_nodes=data.num_nodes,
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
).to(device)

# SISA Method

In [3]:
config['epochs'] = 100
config['num_shards'] = 5
model_list =[]

for _ in range(config['num_shards']):
    model_list.append(LightGCN(
    num_nodes=data.num_nodes,
    embedding_dim=config['embedding_dim'],
    num_layers=config['num_layers'],
).to(device))

shard_models, shards, epoch_tracks, test_topks = sisa_lightgcn_eva(model_list, config, data, device)

IndexError: too many indices for tensor of dimension 1

# Load The Dataset and Retain

In [2]:
from util.dataset_splitter import DatasetSplitter
datasplit = DatasetSplitter(dataset_name="AmazonBook")
retain_data, forget_data = datasplit.load_split()

In [3]:
shard_models = sisa_lightgcn_unlearning_eva(shard_models, shards, retain_data, forget_data, config, device)

NameError: name 'shard_models' is not defined

In [ ]:
sisa_lightgcn_forget_data_eva(shard_models, forget_data, config, device)